# Train the commercial-safe Hebrew handwriting model

This notebook trains the reading model the app uses on its own server, **only on data that allows commercial use**:

| data | what it is | licence |
|---|---|---|
| DiffusionPen Hebrew handwriting (`cyttic/diffusionpen-hebrew-handwriting`) | 116k generated handwriting lines, 491 writing styles | CC-BY-4.0 |
| HHD (`sivan22/hebrew-handwritten-dataset`) | real handwritten Hebrew letters, joined into lines | CC-BY-3.0 |
| Pinkas (zenodo.org/records/3569694) | 943 real handwritten Hebrew lines | CC-BY-4.0 |

It deliberately leaves out IAM, KHATT and BiblIA, whose terms do not allow commercial use. The code is heb-ocr (MIT), pinned to the version the app was tested with.

**How to run it on Kaggle (free):**
1. Create a free account at kaggle.com and verify your phone number (needed for GPUs and internet).
2. *Create → New Notebook → File → Import Notebook*, and upload this file.
3. In the right-hand panel: *Session options → Accelerator: GPU T4 x2* (or P100), and turn *Internet* on.
4. Click **Save Version → Save & Run All (Commit)**. It runs in the background for up to 12 hours (you can close the browser) and keeps everything it writes. Kaggle gives about 30 GPU hours a week.
5. When that version finishes, run it again: *Add Input → Notebook Output Files*, pick this notebook's latest version, then *Save Version* again. The notebook finds the last checkpoint and carries on. Repeat until the log says **TRAINING FINISHED** (about 24 epochs; the log shows the validation error after each one).
6. Download `base.pt`, `hebrew_char6.pkl` and `hebrew_words.pkl` from the last version's output, and put them in the app's `models/` folder on the server.

Credit: heb-ocr by Itay Inbar (MIT); the datasets above (CC-BY: credit their authors where you list sources).

In [ ]:
# Settings
EPOCHS = 24             # full passes over the training mixture; about 1.5-2 GPU hours each on a T4
GLYPH_LINES = 300_000   # lines composed from real handwritten letters, made on the fly
HOURS_THIS_SESSION = 11 # stop safely before Kaggle's 12-hour limit

In [ ]:
%%bash
set -e
if [ ! -d heb-ocr ]; then
  git clone -q https://github.com/itayinbarr/heb-ocr.git
  cd heb-ocr && git checkout -q 19b7c158d14e9293e0806f909f1d497cc4784eaf && cd ..
fi
pip install -q --no-deps -e heb-ocr
pip install -q datasets rapidfuzz opencv-python-headless
mkdir -p heb-ocr/data/hebrew_real
[ -f heb-ocr/data/hebrew_real/pinkas.zip ] || curl -sSL -o heb-ocr/data/hebrew_real/pinkas.zip \
  https://zenodo.org/api/records/3569694/files/pinkas_dataset.zip/content
echo "40857c02c106dcd0e7d7467977c44373dad77a8b680557fbd6f3e560cc42ee6d  heb-ocr/data/hebrew_real/pinkas.zip" | sha256sum -c
nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Pick up where the previous session stopped, if its output was added as an input.
import glob, os, shutil
previous = sorted(glob.glob("/kaggle/input/**/runs/base/last.pt", recursive=True), key=os.path.getmtime)
# Kaggle keeps only /kaggle/working, so the run directory lives there.
if not os.path.islink("heb-ocr/runs"):
    os.makedirs("/kaggle/working/runs/base", exist_ok=True)
    shutil.rmtree("heb-ocr/runs", ignore_errors=True)
    os.symlink("/kaggle/working/runs", "heb-ocr/runs")
resume = []
if previous and not os.path.exists("heb-ocr/runs/base/last.pt"):
    for f in glob.glob(os.path.join(os.path.dirname(previous[-1]), "*")):
        shutil.copy(f, "heb-ocr/runs/base/")
if os.path.exists("heb-ocr/runs/base/last.pt"):
    resume = ["--resume", "runs/base/last.pt"]
print("continuing from the last checkpoint" if resume else "starting from scratch")

In [ ]:
# Train. Stops by itself before the session limit; the checkpoints are kept in the output.
import subprocess, sys
# heb-ocr also scores the ivrit.ai benchmark after every epoch. That dataset
# needs a Hugging Face login, and without one training would stop after the
# first epoch, so this wrapper skips the score when the benchmark can't load.
open("heb-ocr/run_train.py", "w").write('''
import sys
import hebocr.train as T

_score = T.evaluate_benchmark
_told = []

class _Skipped:
    def as_dict(self):
        return {"skipped": True}
    def summary(self):
        return "skipped (benchmark not available)"

def _safe(model, *args, **kwargs):
    try:
        return _score(model, *args, **kwargs)
    except Exception as error:
        model.train()  # the scorer switched it to evaluation mode before failing
        if not _told:
            print(f"benchmark not available ({type(error).__name__}); training goes on without it", flush=True)
            _told.append(1)
        return _Skipped()

T.evaluate_benchmark = _safe
sys.exit(T.main())
''')
args = [sys.executable, "run_train.py", "--out", "runs/base", "--size", "large",
        "--epochs", str(EPOCHS), "--val-limit", "4000",
        "--glyph-lines", str(GLYPH_LINES), "--endless-glyphs",
        "--real-hebrew", "pinkas",          # the only real Hebrew corpus that allows commercial use
        "--real-ink", "",                   # no foreign corpora (some do not allow commercial use)
        "--pixel-budget", "22000", "--concat-prob", "0.35", "--num-workers", "4",
        "--ema", "0.9995", "--seed", "0", *resume]
print(" ".join(args))
try:
    subprocess.run(args, cwd="heb-ocr", check=True, timeout=HOURS_THIS_SESSION * 3600)
    print("TRAINING FINISHED")
except subprocess.TimeoutExpired:
    print("Session time used up. Save the output, start a new session with it as an input, and run again.")

In [ ]:
# The Hebrew letter and word statistics the reader uses, from Hebrew Wikipedia (CC-BY-SA).
# Built once; about 20-40 minutes, on the CPU.
import os, subprocess, sys
for script, out in (("scripts/build_lm.py", "hebrew_char6.pkl"), ("scripts/build_wordlm.py", "hebrew_words.pkl")):
    target = f"/kaggle/working/{out}"
    if not os.path.exists(target):
        previous = [p for p in __import__("glob").glob(f"/kaggle/input/**/{out}", recursive=True)]
        if previous:
            __import__("shutil").copy(previous[0], target)
        else:
            subprocess.run([sys.executable, script, "--out", target], cwd="heb-ocr", check=True)
print("statistics ready")

In [ ]:
# Package the best checkpoint for the app (weights, charset and settings only).
import subprocess, sys, os
if os.path.exists("heb-ocr/runs/base/best.pt"):
    subprocess.run([sys.executable, "scripts/export_model.py", "runs/base/best.pt", "--out", "/kaggle/working/base.pt"],
                   cwd="heb-ocr", check=True)
    print("Download /kaggle/working/base.pt and put it in the app's models/ folder.")

## Optional: measure it on the ivrit.ai benchmark

The benchmark (`ivrit-ai/hebrew-handwriting-ocr-benchmark`) is real handwriting the model never trains on. It needs a free Hugging Face account: accept the dataset's terms on its page, create a read token, and add it under *Add-ons → Secrets* as `HF_TOKEN`.

In [ ]:
import os, subprocess, sys
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret: skipping the benchmark.")
if os.environ.get("HF_TOKEN") and os.path.exists("/kaggle/working/base.pt"):
    subprocess.run([sys.executable, "scripts/make_results.py", "/kaggle/working/base.pt",
                    "--beam-width", "12", "--lm-weight", "0.4", "--out", "/kaggle/working/RESULTS.md"], cwd="heb-ocr")
    print(open("/kaggle/working/RESULTS.md").read())

## Later: improve the shared model with your users' corrections

On the server, `npm run export-training` writes every corrected line from users who allowed it into `training-export.zip`. Upload that zip to Kaggle as a dataset, add it and your `base.pt` as inputs, and run the cell below. It fine-tunes the whole model on those real lines (with the Pinkas lines mixed in so it keeps its general knowledge), and keeps the result only if it reads held-back lines better. Download the new `base.pt`.

In [ ]:
import glob, os, subprocess, sys, zipfile
exports = glob.glob("/kaggle/input/**/training-export.zip", recursive=True)
bases = glob.glob("/kaggle/input/**/base.pt", recursive=True)
if exports and bases:
    with zipfile.ZipFile(exports[0]) as z:
        z.extractall("/kaggle/working/export")
    if not os.path.exists("app"):
        subprocess.run(["git", "clone", "-q", "-b", "claude/hebrew-handwriting-converter-oqrwqk",
                        "https://github.com/gavinissim-cloud/einktablet.git", "app"], check=True)
    os.makedirs("/kaggle/working/models", exist_ok=True)
    subprocess.run(["cp", "heb-ocr/data/hebrew_real/pinkas.zip", "/kaggle/working/models/"], check=True)
    subprocess.run([sys.executable, "app/ocr/train_writer.py", "--models", "/kaggle/working/models",
                    "--base", bases[0], "--examples", "/kaggle/working/export/lines.tsv",
                    "--out", "/kaggle/working/base-new.pt", "--device", "cuda", "--top-layers", "8",
                    "--max-steps", "3000", "--steps", "0"], check=True)
    print("If it says kept: true, download /kaggle/working/base-new.pt and use it as the new base.pt.")
else:
    print("Add training-export.zip and base.pt as inputs to run this.")